# Phases block extraction and comparison

First step is to use and tweak Joshua's code to get the phase block instead of solution species block.

In [1]:
import warnings
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import build_database.clean_tables as ct
import build_database.utils as ut
import build_database.write_dataframes as wd
import utils
import importlib.resources as pkg_resources
import build_database.parser_dat as parser_dat

Josh had made a function for extracting different phases, so will check if this works alright. I'll switch off the part of the code that removes the duplicates, since we need to see the number of each databases' phases correctly.

In [2]:
database_list = pkg_resources.files('build_database.databases')
database_list = ut.phreeqc_database_list(database_list)
phases = ct.compile_phase_table(database_list)
display(phases)


,phase_name,dissolution_reaction,log_k,delta_h,analytic,v_m,t_c,p_c,omega,source
0,(UO2)2As2O7,(UO2)2As2O7 +2H+ +H2O = + 2H2AsO4- + 2UO2+2,None,None,None,None,None,None,None,llnl.dat
1,(UO2)2Cl3,(UO2)2Cl3 = + UO2+ + UO2+2 + 3Cl-,None,None,None,None,None,None,None,llnl.dat
2,(UO2)2P2O7,(UO2)2P2O7 +H2O = + 2HPO4-2 + 2UO2+2,None,None,None,None,None,None,None,llnl.dat
3,(UO2)3(AsO4)2,(UO2)3(AsO4)2 +4H+ = + 2H2AsO4- + 3UO2+2,None,None,None,None,None,None,None,llnl.dat
4,(UO2)3(PO4)2,(UO2)3(PO4)2 +2H+ = + 2HPO4-2 + 3UO2+2,None,None,None,None,None,None,None,llnl.dat
...,...,...,...,...,...,...,...,...,...,...
3490,H2(g),H2 = H2,None,None,None,None,None,None,None,wateq4f.dat
3491,N2(g),N2 = N2,None,None,None,None,None,None,None,wateq4f.dat
3492,H2S(g),H2S = H2S,None,None,None,None,None,None,None,wateq4f.dat
3493,CH4(g),CH4 = CH4,None,None,None,None,None,None,None,wateq4f.dat


Let's test it one by one if it contains all the phases in each database file.

In [3]:
# checking how many phases llnl has
phases_llnl = phases[phases['source'].str.contains('llnl')]
display(phases_llnl)
#llnl contains 1215 phases according to this.

,phase_name,dissolution_reaction,log_k,delta_h,analytic,v_m,t_c,p_c,omega,source
0,(UO2)2As2O7,(UO2)2As2O7 +2H+ +H2O = + 2H2AsO4- + 2UO2+2,None,None,None,None,None,None,None,llnl.dat
1,(UO2)2Cl3,(UO2)2Cl3 = + UO2+ + UO2+2 + 3Cl-,None,None,None,None,None,None,None,llnl.dat
2,(UO2)2P2O7,(UO2)2P2O7 +H2O = + 2HPO4-2 + 2UO2+2,None,None,None,None,None,None,None,llnl.dat
3,(UO2)3(AsO4)2,(UO2)3(AsO4)2 +4H+ = + 2H2AsO4- + 3UO2+2,None,None,None,None,None,None,None,llnl.dat
4,(UO2)3(PO4)2,(UO2)3(PO4)2 +2H+ = + 2HPO4-2 + 3UO2+2,None,None,None,None,None,None,None,llnl.dat
...,...,...,...,...,...,...,...,...,...,...
1210,UOF4(g),UOF4 +H2O = + UO2+2 + 2H+ + 4F-,None,None,None,None,None,None,None,llnl.dat
1211,Xe(g),Xe = + Xe,None,None,None,None,None,None,None,llnl.dat
1212,Zn(g),Zn +2H+ +0.5000 O2 = + H2O + Zn+2,None,None,None,None,None,None,None,llnl.dat
1213,Zr(g),Zr +4H+ +O2 = + Zr+4 + 2H2O,None,None,None,None,None,None,None,llnl.dat


In [4]:
#So I copied and pasted whole 'phases' section from llnl and made a simple code to count the lines that don't start with # or a tab.
phasefile = open("llnl_phase.txt","r")
outputphase = open("llnl_phasename.txt","w")
all_lines = phasefile.readlines()
for i in range(0,len(all_lines)):
    line = all_lines[i]
    
    if line != "" and line[0] != '#' and line[0] != '\t' and line[0] != '\n' and line[0] != ' ':
        outputphase.write(all_lines[i])

phasefile.close()
outputphase.close()
# the result showed that there are 1215 phases in phases block

Let's see if these ones are correct

In [5]:
outputphase = open("llnl_phasename.txt",'r')
all_lines = outputphase.readlines()
for i in range(0,len(all_lines)):
    line = all_lines[i].strip()
    line_clean = line.strip()
    name_clean = phases_llnl['phase_name'].str.strip()
    name_clean.to_csv("phase_names.txt", index=False, header=False)


    if not (name_clean == line_clean).any():
        print(line)

outputphase.close()
# seems like it contains the correct ones!

Let's now check how many phases other databases have

In [6]:
# checking how many phases other databases have
phases_sit = phases[phases['source'].str.contains('sit')]
print(len(phases_sit))
#sit contains 862 phases according to this.
phases_minteq = phases[phases['source'].str.contains('minteq')]
print(len(phases_minteq))
#minteq.v4 contains 645 phases according to this.
phases_phreeqc = phases[phases['source'].str.contains('phreeqc')]
print(len(phases_phreeqc))
#phreeqc contains 71 phases according to this.
phases_pitzer = phases[phases['source'].str.contains('pitzer')]
print(len(phases_pitzer))
#pitzer contains 72 phases according to this.

862
645
71
72


1. SIT

In [7]:
phasefile = open("sit_phase.txt","r")
outputphase = open("sit_phasename.txt","w")
all_lines = phasefile.readlines()
count = 0
for i in range(0,len(all_lines)):
    line = all_lines[i]
    
    if line != "" and line[0] != '#' and line[0] != '\t' and line[0] != '\n' and line[0] != ' ' and '=' not in line:
        outputphase.write(all_lines[i])
        count += 1

print(count)
phasefile.close()
outputphase.close()

# number of phases is 863

863


In [8]:
outputphase = open("sit_phasename.txt",'r')
sit_lines = [line.strip() for i, line in enumerate(outputphase)]
outputphase.close()

name_clean = phases_sit['phase_name'].str.strip()

missing_in_df = [line for line in sit_lines if line not in name_clean.values]
print("Exists in the database but filtered")
print(missing_in_df)

missing_in_sit = [name for name in name_clean if name not in sit_lines]
print("Exists in the database but not added")
print(missing_in_sit)


Exists in the database but filtered
['MgBr2(s)']
Exists in the database but not added
[]


2. minteq.v4

In [9]:
phasefile = open("minteq_phase.txt","r")
outputphase = open("minteq_phasename.txt","w")
all_lines = phasefile.readlines()
count = 0
for i in range(0,len(all_lines)):
    line = all_lines[i]
    
    if line != "" and line[0] != '#' and line[0] != '\t' and line[0] != '\n' and line[0] != ' ':
        outputphase.write(all_lines[i])
        count += 1

print(count)
phasefile.close()
outputphase.close()


568


In [10]:
outputphase = open("minteq_phasename.txt",'r')
minteq_lines = [line.strip() for i, line in enumerate(outputphase)]
outputphase.close()

name_clean = phases_minteq['phase_name'].str.strip()

missing_in_df = [line for line in minteq_lines if line not in name_clean.values]
print("Exists in the database but filtered")
print(missing_in_df)

missing_in_minteq = [name for name in name_clean if name not in minteq_lines]
print("Exists in the database but not added")
print(missing_in_minteq)

Exists in the database but filtered
[]
Exists in the database but not added
['log_k 0.0', 'log_k 0.0', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ', 'delta_h\t0\tkJ

3. phreeqc

In [11]:
phasefile = open("phreeqc_phase.txt","r")
outputphase = open("phreeqc_phasename.txt","w")
all_lines = phasefile.readlines()
count = 0

for i in range(0,len(all_lines)):
    line = all_lines[i]
    
    if line != "" and line[0] != '#' and line[0] != '\t' and line[0] != '\n' and line[0] != ' ':
        outputphase.write(all_lines[i])
        count += 1

print(count)
phasefile.close()
outputphase.close()
# number of phases is 71(correct)

71


4. pitzer

In [12]:
phasefile = open("pitzer_phase.txt","r")
outputphase = open("pitzer_phasename.txt","w")
all_lines = phasefile.readlines()
count = 0

for i in range(0,len(all_lines)):
    line = all_lines[i]
    
    if line != "" and line[0] != '#' and line[0] != '\t' and line[0] != '\n' and line[0] != ' ':
        outputphase.write(all_lines[i])
        count += 1

print(count)
phasefile.close()
outputphase.close()
# number of phases is 72(correct)

72


# Fixing the code further
Josh's code doesn't catch phases with the space in it, I fixed it by using 
self.block_start = re.compile(r"((^(?!\s)\S+$)|(^((?!.*=).)+$))")

class PhaseParser(BaseParser):
    """Class for parsing phase species in phreeqc database files"""

    def __init__(self, database_file_path) -> None:
        """
        Initialize the PhaseParser class.

        Parameters:
        - database_file_path (str): The path to the database file.
        """
        super().__init__(database_file_path, "PHASES", "EXCHANGE")
        # momentarily changed it to a new bit to test if it catches the missing part
        #self.block_start = re.compile(r"^(?!\s)\S+$")
        self.block_start = re.compile(r"^(?!.*=).+$")
        self.patterns = {
            "dissolution_reaction": re.compile(r"^.*\s=\s.*"),
            "log_k": re.compile(r"^\s*[-]*log[ _]*k"),
            "delta_h": re.compile(r"\s*[-]*delta.*"),
            "analytic": re.compile(r"^\s*[-]*analytic"),
            "v_m": re.compile(r"^\s*[-]*Vm"),
            "t_c": re.compile(r"^\s*[-]*T_c"),
            "p_c": re.compile(r"^\s*[-]*P_c"),
            "omega": re.compile(r"^\s*[-]*Omega"),
        }

But this resulted in a fault in minteq(which is due to minteq's different database system compared to others), and also it rules out MgBr(s)